# get historical monthly urls 

In [0]:
import requests
import re
from urllib.parse import urljoin

def build_monthly_url(year, month):
    folder = (
        "https://www.nemweb.com.au/"
        "Data_Archive/Wholesale_Electricity/MMSDM/"
        f"{year}/MMSDM_{year}_{month:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    response = requests.get(folder)

    # Month not published yet
    if response.status_code == 404:
        return None

    response.raise_for_status()

    files = re.findall(
        r'[^"\s<>]*DISPATCHREGIONSUM[^"\s<>]*\.zip',
        response.text,
        flags=re.IGNORECASE
    )

    files = [
        file for file in files
        if "PREDISPATCH" not in file.upper()
    ]

    if not files:
        return None

    return urljoin(folder, files[0])

In [0]:
url = build_monthly_url(2015, 1)
print(url)

In [0]:
from datetime import date
import time

today = date.today()

urls = []

start = time.time()

for year in range(2015, today.year + 1):
    for month in range(1, 13):

        if (year, month) >= (today.year, today.month):
            continue

        month_start = time.time()

        print(f"Checking {year}-{month:02d}...", end=" ")

        url = build_monthly_url(year, month)

        elapsed = time.time() - month_start

        if url:
            urls.append(url)
            print(f"found ({elapsed:.1f}s)")
        else:
            print(f"not published ({elapsed:.1f}s)")

total_time = time.time() - start

print()
print(f"Found {len(urls)} files")
print(f"Total time: {total_time:.1f} seconds")

In [0]:
urls

# download files from urls 

In [0]:
import os
import requests
import time

def download_if_not_exists(url, bronze_folder):
    filename = url.split("/")[-1]
    path = os.path.join(bronze_folder, filename)

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")
    start = time.time()

    response = requests.get(url)
    response.raise_for_status()

    with open(path, "wb") as file:
        file.write(response.content)

    print(
        f"Finished: {filename} - "
        f"{time.time() - start:.1f} seconds"
    )

    return path

In [0]:
bronze_folder = "/Volumes/workspace/default/aemo_mlops_volume/bronze/monthly"

os.makedirs(bronze_folder, exist_ok=True)

for url in urls:
    download_if_not_exists(url, bronze_folder)

# uncompress files

In [0]:
import os
import zipfile
import time

csv_folder = bronze_folder + "_uncompressed"

os.makedirs(csv_folder, exist_ok=True)

zip_files = [
    file for file in os.listdir(bronze_folder)
    if file.endswith(".zip")
]

start = time.time()

print(f"Found {len(zip_files)} ZIP files\n")

for i, filename in enumerate(sorted(zip_files), 1):

    path = os.path.join(bronze_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:

        files = zip_file.namelist()

        already_extracted = all(
            os.path.exists(os.path.join(csv_folder, file))
            for file in files
        )

        if already_extracted:
            print(f"[{i}/{len(zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(zip_files)}] Extracting: {filename}")

        zip_file.extractall(csv_folder)

print()
print(f"Finished in {time.time() - start:.1f} seconds")